# UNTR Sentiment-Window Model

`Stock UT pred.ipynb` trains on the full ~1-year price history, where news sentiment only covers the last few months -- 98% of the training rows are structurally zero for the sentiment columns, so that model can't say anything about whether sentiment helps.

This notebook does one thing: **train and evaluate using only the dates where UNTR news sentiment actually exists.** Same features, same honest baseline-comparison method as the main notebook -- just restricted to the window where the sentiment features have real values to learn from.

Rerun this notebook whenever `data/news/UNTR/untr_news_sentiment.csv` is refreshed (via `UNTR News Sentiment.ipynb`) -- the news-covered window grows over time, so the comparison gets more statistical power for free as more days accumulate.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error

## Load price data

In [2]:
df = pd.read_csv("../data/prices/UNTR.csv")
df_close = df[["Date", "Close"]].copy()
df_close.index = pd.to_datetime(df_close["Date"], format="%Y-%m-%d")
df_close = df_close.drop(columns="Date")

print(f"Price history: {df_close.index.min().date()} to {df_close.index.max().date()} ({len(df_close)} trading days)")

Price history: 2025-09-26 to 2026-09-25 (244 trading days)


## Load news sentiment and aggregate to daily

In [3]:
news = pd.read_csv("../data/news/UNTR/untr_news_sentiment.csv")
news["date"] = pd.to_datetime(news["timestamp"]).dt.normalize()

# unclear/error contribute 0 rather than being dropped -- "no clear signal
# that day" is itself informative, not missing.
sentiment_map = {"positive": 1, "negative": -1, "unclear": 0, "error": 0}
news["sentiment_num"] = news["sentiment"].map(sentiment_map).fillna(0)

daily_sentiment = news.groupby("date").agg(
    sentiment_sum=("sentiment_num", "sum"),
    article_count=("sentiment_num", "count"),
).sort_index()

print(f"News pulled: {len(news)} articles, {news['date'].min().date()} to {news['date'].max().date()}")
print(f"Unique calendar dates with news: {len(daily_sentiment)}")

News pulled: 113 articles, 2026-05-30 to 2026-09-26
Unique calendar dates with news: 47


## Restrict to the sentiment-covered window

Reindex daily sentiment onto the trading-day index (news on a weekend/holiday has no matching trading day and is dropped here -- there's no next-session rollover in this version), then slice `df_close` down to only the trading days from the first day sentiment coverage starts.

In [4]:
daily_sentiment = daily_sentiment.reindex(df_close.index, fill_value=0)
df_close["sentiment_sum"] = daily_sentiment["sentiment_sum"]
df_close["article_count"] = daily_sentiment["article_count"]

news_window_start = df_close.index[df_close["article_count"] > 0].min()
df_close = df_close.loc[df_close.index >= news_window_start].copy()

print(f"Sentiment-covered window: {df_close.index.min().date()} to {df_close.index.max().date()} ({len(df_close)} trading days)")
print(f"Trading days with any news in this window: {(df_close['article_count'] > 0).sum()} / {len(df_close)}")

Sentiment-covered window: 2026-06-03 to 2026-09-25 (81 trading days)
Trading days with any news in this window: 41 / 81


## Feature engineering (same conventions as `Stock UT pred.ipynb`)

In [5]:
df_close["dayofweek"] = df_close.index.dayofweek

# Return target + return-based features (model the % change, not price level)
df_close["ret"] = df_close["Close"].pct_change()
df_close["ret_lag1"] = df_close["ret"].shift(1)
df_close["ret_lag2"] = df_close["ret"].shift(2)
df_close["ret_lag3"] = df_close["ret"].shift(3)
df_close["ret_lag5"] = df_close["ret"].shift(5)
df_close["vol_5"] = df_close["ret"].shift(1).rolling(5).std()

# Sentiment features -- shift(1) so today's row never sees today's own news
df_close["sentiment_lag1"] = df_close["sentiment_sum"].shift(1)
df_close["article_count_lag1"] = df_close["article_count"].shift(1)
df_close["sentiment_roll3"] = df_close["sentiment_sum"].shift(1).rolling(3).sum()

RET_FEATURES = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5", "vol_5", "dayofweek"]
SENT_FEATURES = ["sentiment_lag1", "article_count_lag1", "sentiment_roll3"]
RET_FEATURES_SENT = RET_FEATURES + SENT_FEATURES
TARGET = "ret"

df_close.tail()

,Close,sentiment_sum,article_count,dayofweek,ret,ret_lag1,ret_lag2,ret_lag3,ret_lag5,vol_5,sentiment_lag1,article_count_lag1,sentiment_roll3
Date,,,,,,,,,,,,,
2026-09-21,24700.0,-1,1,0,-0.005035,-0.034047,-0.014382,-0.000958,0.003802,0.014679,1.0,1.0,1.0
2026-09-22,24325.0,-2,3,1,-0.015182,-0.005035,-0.034047,-0.014382,-0.011364,0.012805,-1.0,1.0,0.0
2026-09-23,24400.0,-2,14,2,0.003083,-0.015182,-0.005035,-0.034047,-0.000958,0.012785,-2.0,3.0,-2.0
2026-09-24,24150.0,0,0,3,-0.010246,0.003083,-0.015182,-0.005035,-0.014382,0.013890,-2.0,14.0,-5.0
2026-09-25,24225.0,0,0,4,0.003106,-0.010246,0.003083,-0.015182,-0.034047,0.013918,0.0,0.0,-4.0


## Train / validation / test split

Chronological split within the sentiment-covered window only. Small sample by construction -- treat any result here as a first, low-power read, not a confident verdict.

In [6]:
N_TEST = 20
N_VAL = 15

data = df_close.dropna(subset=RET_FEATURES_SENT + [TARGET])
test = data.iloc[-N_TEST:]
train_full = data.iloc[:-N_TEST]
val = train_full.iloc[-N_VAL:]
train = train_full.iloc[:-N_VAL]

print(f"train / val / test : {len(train)} / {len(val)} / {len(test)}")
print(f"test window        : {test.index.min().date()} to {test.index.max().date()}")

train / val / test : 40 / 15 / 20
test window        : 2026-08-31 to 2026-09-25


## Baseline: naive persistence

In [7]:
baseline_pred = df_close["Close"].shift(1).loc[test.index]
baseline_rmse = np.sqrt(mean_squared_error(test["Close"], baseline_pred))
print(f"Naive persistence baseline RMSE: {baseline_rmse:.2f}")

Naive persistence baseline RMSE: 460.60


## Train and score: with vs. without sentiment

In [8]:
def fit_and_score(features, label):
    X_tr, y_tr = train[features], train[TARGET]
    X_val, y_val = val[features], val[TARGET]
    X_te = test[features]

    model = xgb.XGBRegressor(n_estimators=1000, early_stopping_rounds=50, learning_rate=0.01, max_depth=3)
    model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)

    pred_ret = model.predict(X_te)
    prev_close = df_close["Close"].shift(1).loc[test.index]
    pred_close = prev_close * (1 + pred_ret)
    rmse = np.sqrt(mean_squared_error(test["Close"], pred_close))
    fi = pd.Series(model.feature_importances_, index=model.feature_names_in_).sort_values(ascending=False)

    print(f"[{label}] RMSE: {rmse:.2f}  |  best_iteration: {model.best_iteration}")
    return model, rmse, fi, pred_close


model_base, rmse_base, fi_base, pred_close_base = fit_and_score(RET_FEATURES, "no sentiment")
model_sent, rmse_sent, fi_sent, pred_close_sent = fit_and_score(RET_FEATURES_SENT, "with sentiment")

print()
print(f"Naive baseline RMSE      : {baseline_rmse:.2f}")
print(f"Return model, no sentiment: {rmse_base:.2f}")
print(f"Return model, w/ sentiment: {rmse_sent:.2f}")
print()
print("Feature importances (with sentiment):")
print(fi_sent)

[no sentiment] RMSE: 462.40  |  best_iteration: 0
[with sentiment] RMSE: 462.96  |  best_iteration: 0

Naive baseline RMSE      : 460.60
Return model, no sentiment: 462.40
Return model, w/ sentiment: 462.96

Feature importances (with sentiment):
sentiment_roll3       0.182095
ret_lag2              0.152802
article_count_lag1    0.142635
sentiment_lag1        0.134529
ret_lag3              0.131474
vol_5                 0.127653
ret_lag1              0.100547
ret_lag5              0.016493
dayofweek             0.011773
dtype: float32


## Reading this result

Compare the three RMSEs above (baseline vs. no-sentiment model vs. sentiment model), and check `best_iteration` for each model -- a value of `0` means early stopping judged the very first boosting round as the best one, which with a small `train` set usually means there isn't enough data for the model to learn *anything* reliably (sentiment or otherwise), not that sentiment specifically failed. Don't read a tie or a small win/loss here as a final verdict on sentiment; the sample is small by construction. The number that matters most for deciding whether to trust this comparison at all is `len(train)` above -- as `data/news/UNTR/untr_news_sentiment.csv` accumulates more days (via `UNTR News Sentiment.ipynb`), rerun this notebook and watch that number, and this result, become more trustworthy.